In [ ]:
import torch 
import torch.nn as nn

class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_len, dropout=0.1):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out

        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))
        
        self.w_q = nn.Linear(d_in, d_out)
        self.w_k = nn.Linear(d_in, d_out)
        self.w_v = nn.Linear(d_in, d_out)

        self.out_proj = nn.Linear(d_out, d_out)

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.w_q(x)
        keys = self.w_k(x)
        values = self.w_v(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        #attn_output = torch.nn.functional.scaled_dot_product_attention(
        #    queries,
        #    keys,
        #    values,
        #    attn_mask=self.mask ,
        #    dropout_p=self.dropout.p if self.training else 0.0,
        #    is_causal=True
        #)

        attn_scores = torch.matmul(queries, keys.transpose(-2, -1)) / (self.head_dim ** 0.5)
        mask = self.mask[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask == 1, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, values)

        attn_output = attn_output.transpose(1,2).contiguous().view(b, num_tokens, self.d_out)
        
        return self.out_proj(attn_output)
